<a href="https://colab.research.google.com/github/algroznykh/closed_form_nca/blob/main/closed_form_spectral_featureviz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# -*- coding: utf-8 -*-
"""
Modular Closed-Form NCA with Euler Self-Organizing Inference
Enhanced with truly complex unitary mixing and scale-dependent spectral viscosity.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import cv2
import io
import numpy as np
import traceback
import ipywidgets as widgets
from IPython.display import display, HTML
from collections import deque
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Colab-specific interface module
from google.colab import output

torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- SECTION 1: CLIP INITIALIZATION & ACTIVATION HOOKS ---
try:
    import clip
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "git+https://github.com/openai/CLIP.git"])
    import clip

clip_model, _ = clip.load('RN101', device=device, jit=False)
clip_model.eval().float()
for p in clip_model.parameters():
    p.requires_grad_(False)

CLIP_MEAN = torch.tensor([0.4814, 0.4578, 0.4082], device=device).view(1, 3, 1, 1)
CLIP_STD  = torch.tensor([0.2686, 0.2613, 0.2758], device=device).view(1, 3, 1, 1)

def clip_norm(x):
    return (x - CLIP_MEAN) / CLIP_STD

_act = {}
def get_hook(name):
    def _hook(mod, inp, out):
        _act[name] = out
    return _hook

for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
    dict(clip_model.visual.named_modules())[layer_name].register_forward_hook(get_hook(layer_name))


# --- SECTION 2: CLOSED-FORM NCA WITH COMPLEX ARITHMETIC ---

class ClosedFormNCA(nn.Module):
    """
    Closed-Form NCA with truly complex unitary mixing and spectral viscosity
    to balance energy across spatial frequencies.
    """
    def __init__(self, in_ch=12, latent_ch=48, kernel_size=5, projector_type="spatial"):
        super().__init__()
        self.C = latent_ch
        self.k_size = kernel_size
        self.shift = kernel_size // 2

        self.lift = nn.Sequential(
            nn.Conv2d(in_ch, latent_ch, 1), nn.GELU(),
            nn.Conv2d(latent_ch, latent_ch, 1)
        )

        # Robust spatial projection head
        self.project = nn.Sequential(
            nn.Conv2d(latent_ch, latent_ch, kernel_size=3, padding=1, padding_mode='circular'),
            nn.GELU(),
            nn.Conv2d(latent_ch, 3, kernel_size=1),
            nn.Sigmoid()
        )

        # Parameters for constructing a truly complex unitary matrix
        self.U_real = nn.Parameter(torch.randn(latent_ch, latent_ch) * 0.05)
        self.U_imag = nn.Parameter(torch.randn(latent_ch, latent_ch) * 0.05)

        # Spatial kernel with active growth characteristics
        self.spatial_kernel = nn.Parameter(torch.randn(latent_ch, 1, kernel_size, kernel_size) * 0.15)

        # Positive center-bias to trigger local propagation
        with torch.no_grad():
            self.spatial_kernel[:, 0, self.shift, self.shift] += 0.2

    def _get_unitary_matrix(self):
        """Constructs a truly complex unitary matrix via a skew-Hermitian generator."""
        A = self.U_real - self.U_real.T  # Skew-symmetric
        B = self.U_imag + self.U_imag.T  # Symmetric
        M = torch.complex(A, B)          # Skew-Hermitian generator (M^H = -M)
        return torch.matrix_exp(M)       # Exp(M) is unitary (U^H * U = I)

    def forward(self, t, x0):
        z0 = self.lift(x0)
        orig_dtype = z0.dtype
        H, W = x0.shape[2], x0.shape[3]

        # Truly complex unitary mixing matrix
        U_complex = self._get_unitary_matrix()

        # Compute frequency response of spatial kernel
        pad_w, pad_h = W - self.k_size, H - self.k_size
        k_pad = torch.roll(F.pad(self.spatial_kernel, (0, pad_w, 0, pad_h)), (-self.shift, -self.shift), (2, 3))
        g_hat = torch.fft.fft2(k_pad).squeeze(1)

        # 2D Frequency Coordinates squared (|k|^2) for spectral viscosity
        freq_y = torch.fft.fftfreq(H, device=x0.device).view(1, H, 1)
        freq_x = torch.fft.fftfreq(W, device=x0.device).view(1, 1, W)
        k_sq = freq_y**2 + freq_x**2

        # Combine active growth with spectral viscosity (dampens high frequencies)
        viscosity_coeff = 0.35
        g_real = torch.tanh(g_hat.real * 1.0) * 0.65 - (viscosity_coeff * k_sq)
        g_imag = g_hat.imag
        g_stable = g_real + 1j * g_imag

        # Rotate, evolve over continuous time step, and rotate back
        Z0 = torch.fft.fft2(z0.to(torch.float32))
        Z_rot = torch.einsum('cd, bdhw -> bchw', U_complex.conj().T, Z0)

        evolution = torch.exp(t.to(torch.float32) * g_stable)
        Z_t = Z_rot * evolution

        # Point-wise spectral magnitude scaling to protect gradient stability
        mag = torch.abs(Z_t)
        scale = torch.tanh(mag / 12.0) * 12.0 / (mag + 1e-8)
        Z_t = Z_t * scale

        Z_final = torch.einsum('cd, bdhw -> bchw', U_complex, Z_t)
        xt = torch.fft.ifft2(Z_final).real.to(orig_dtype)

        # Bounding activation
        xt = torch.tanh(xt)

        return xt, self.project(xt)

    def step_latent_euler(self, z, dt=0.05):
        """Euler step utilizing the complex unitary mixing matrix."""
        orig_dtype = z.dtype
        z_f32 = z.to(torch.float32)

        k_conv = torch.flip(self.spatial_kernel, [2, 3])
        U_complex = self._get_unitary_matrix()

        # Extract the real part of the unitary matrix for real-valued spatial execution
        U_real = U_complex.real

        def dynamics(x):
            x_rot = F.pad(torch.einsum('cd, bdhw -> bchw', U_real.T, x),
                          (self.shift, self.shift, self.shift, self.shift), mode='circular')
            dx_rot = F.conv2d(x_rot, k_conv, groups=self.C)
            return torch.einsum('cd, bdhw -> bchw', U_real, torch.tanh(dx_rot))

        # Euler step with self-limiting boundary
        z_new = z_f32 + dt * dynamics(z_f32)
        return torch.tanh(z_new).to(orig_dtype)


# --- SECTION 3: TASK REPRESENTATION ---

class BaseTask:
    def generate_input(self, batch_size, channels, size, device):
        raise NotImplementedError
    def compute_loss(self, rgb, config):
        raise NotImplementedError


class CLIPFeatureTargetTask(BaseTask):
    """CLIP feature and DeepDream optimization task."""
    def generate_input(self, batch_size, channels, size, device):
        noise = torch.randn(batch_size, channels, size, size, device=device) * 0.15
        return noise

    def _random_crops(self, img, n=2, size=224):
        B, C, H, W = img.shape
        scales = torch.empty(n, device=img.device).uniform_(0.8, 1.1)
        tx = torch.empty(n, device=img.device).uniform_(-0.05, 0.05)
        ty = torch.empty(n, device=img.device).uniform_(-0.05, 0.05)
        flip = (torch.rand(n, device=img.device) < 0.5).float() * 2 - 1

        theta = torch.zeros(n, 2, 3, device=img.device)
        theta[:, 0, 0] = scales * flip
        theta[:, 1, 1] = scales
        theta[:, 0, 2] = tx
        theta[:, 1, 2] = ty

        grid = F.affine_grid(theta, (n, C, size, size), align_corners=False)
        expanded = img.unsqueeze(1).expand(-1, n, -1, -1, -1).reshape(B * n, C, H, W)
        return F.grid_sample(expanded, grid.repeat(B, 1, 1, 1), padding_mode='reflection', align_corners=False)

    def compute_loss(self, rgb, config):
        views = self._random_crops(rgb, n=2)
        views = views + torch.randn_like(views) * 0.01

        _ = clip_model.encode_image(clip_norm(views))
        current_layer = config.get("target_layer", "layer2")
        current_acts = _act[current_layer]

        if config.get("is_deep_dream", False):
            fv_loss = -current_acts.square().mean() * 6.0
        else:
            target_ch = config.get("target_channel", 42)
            fv_loss = -current_acts[:, target_ch].mean() * 6.0

        l2_reg = (rgb - 0.5).square().mean() * 0.1
        tv_loss = ((rgb[:, :, :, :-1] - rgb[:, :, :, 1:]).square().mean() +
                   (rgb[:, :, :-1, :] - rgb[:, :, 1:, :]).square().mean()) * 0.05

        # Activity Loss targeting structural variance
        spatial_std = rgb.std(dim=(2, 3)).mean()
        activity_loss = (spatial_std - 0.22).square() * 15.0

        # Gentle spectral consolidation
        Z_loss = torch.fft.rfft2(rgb.float())
        mag = torch.abs(Z_loss)
        mag_ac = mag.clone()
        mag_ac[:, :, 0, 0] = 0.0
        mag_sum = mag_ac.sum(dim=(2, 3), keepdim=True) + 1e-6
        mag_norm = mag_ac / mag_sum
        entropy = - (mag_norm * torch.log(mag_norm + 1e-8)).sum(dim=(2, 3)).mean()
        spec_consolidation = entropy * 0.05

        total_loss = fv_loss + l2_reg + tv_loss + activity_loss + spec_consolidation
        return total_loss, {
            "total": total_loss.item(),
            "feature": fv_loss.item(),
            "l2": l2_reg.item(),
            "tv": tv_loss.item(),
            "sparsity": activity_loss.item()
        }


# --- SECTION 4: SWAPPABLE TRAINING REGIME ---

class NCATrainer:
    def __init__(self, model, task, lr=4e-3, weight_decay=1e-4):
        self.model = model
        self.task = task
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        self.scaler = torch.amp.GradScaler('cuda')

        self.logs = {
            "total": [],
            "feature": [],
            "l2": [],
            "tv": [],
            "sparsity": []
        }
        self.best_loss = float('inf')
        self.train_times = deque(maxlen=30)

    def step(self, config):
        t_start = time.time()

        t_train = torch.empty(2, 1, 1, 1, device=device).uniform_(4.0, 16.0)
        x0 = self.task.generate_input(batch_size=2, channels=12, size=128, device=device)

        with torch.amp.autocast('cuda'):
            _, rgb = self.model(t_train, x0)
            loss, loss_breakdown = self.task.compute_loss(rgb, config)

        self.optimizer.zero_grad()
        self.scaler.scale(loss).backward()
        self.scaler.unscale_(self.optimizer)

        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()

        for k, v in loss_breakdown.items():
            if k in self.logs:
                self.logs[k].append(v)

        feat_val = loss_breakdown["feature"]
        if feat_val < self.best_loss:
            self.best_loss = feat_val

        self.train_times.append(time.time() - t_start)
        return loss_breakdown

    def reset_logs(self):
        for k in self.logs:
            self.logs[k].clear()
        self.best_loss = float('inf')


# --- SECTION 5: MODULAR UI & CONTROLLER SYSTEM ---

class Dashboard:
    def __init__(self, model_class, task_class, trainer_class):
        self.model_class = model_class
        self.task_class = task_class
        self.trainer_class = trainer_class
        self.projector_type = "spatial"
        self.init_system()

        # Build individual control widgets
        self.layer_dropdown = widgets.Dropdown(
            options=['layer1', 'layer2', 'layer3', 'layer4'],
            value='layer2', description='Layer:', layout=widgets.Layout(width='180px')
        )
        self.channel_slider = widgets.IntSlider(
            min=0, max=511, value=42, description='Channel:', layout=widgets.Layout(width='340px')
        )
        self.dd_toggle = widgets.Checkbox(
            value=False, description='DeepDream', layout=widgets.Layout(width='120px')
        )
        self.btn_hard_reset = widgets.Button(
            description='💣 Hard Reset Model', button_style='danger', icon='bomb', layout=widgets.Layout(width='180px')
        )
        self.zoom_slider = widgets.IntSlider(
            min=384, max=1536, value=768, step=96, description='Scale Viewport:', layout=widgets.Layout(width='300px')
        )
        self.btn_record = widgets.ToggleButton(
            description='🔴 Record Video', button_style='info', icon='video-camera', layout=widgets.Layout(width='150px')
        )
        self.proj_toggle = widgets.Dropdown(
            options=['spatial'], value='spatial', description='Projection:', layout=widgets.Layout(width='180px')
        )

        self.img_widget = widgets.Image(
            value=cv2.imencode('.jpg', np.zeros((128, 384, 3), np.uint8))[1].tobytes(),
            format='jpeg'
        )
        self.img_widget.add_class("colab-nca-viewport")
        self.img_widget.layout.width = "768px"
        self.img_widget.layout.height = "256px"

        self.fps_label = widgets.Label(value='Ready')
        self.stats_label = widgets.HTML(
            value="Waiting for metrics...",
            layout=widgets.Layout(margin='0 0 0 15px')
        )

        self.graph_widget = widgets.Image(format='png', width=500, height=200)
        self._update_graph_placeholder("Waiting for loss data...")

        self.is_running = True
        self.is_paused = False
        self.t = 0.0
        self._pause_offset = 0.0
        self._unpause_wall = time.time()
        self.recorded_frames = []

        self._setup_handlers()

    def init_system(self):
        self.model = self.model_class(
            in_ch=12, latent_ch=48, kernel_size=5, projector_type=self.projector_type
        ).to(device)
        self.task = self.task_class()
        self.trainer = self.trainer_class(self.model, self.task)

        self.display_x0 = self.task.generate_input(batch_size=1, channels=12, size=128, device=device)
        self.z_rk4 = self.model.lift(self.display_x0)

    def _setup_handlers(self):
        self.layer_dropdown.observe(self._on_layer_change, 'value')
        self.channel_slider.observe(self._on_loss_param_change, 'value')
        self.dd_toggle.observe(self._on_dd_toggle, 'value')
        self.proj_toggle.observe(self._on_projection_type_change, 'value')
        self.btn_hard_reset.on_click(self._on_hard_reset_click)
        self.zoom_slider.observe(self._on_zoom_change, 'value')
        self.btn_record.observe(self._on_record_toggle, 'value')

        self.btn_toggle = widgets.ToggleButton(value=True, icon='stop', button_style='danger', layout=widgets.Layout(width='32px'))
        self.btn_toggle.observe(self._on_play_stop_toggle, names='value')
        self.btn_pause = widgets.ToggleButton(icon='pause', button_style='warning', layout=widgets.Layout(width='32px'))
        self.btn_pause.observe(self._on_pause_toggle, names='value')
        self.btn_reset = widgets.Button(icon='refresh', button_style='info', layout=widgets.Layout(width='32px'))
        self.btn_reset.on_click(lambda _: self.reset_time())

    def _on_layer_change(self, change):
        ch_map = {'layer1': 256, 'layer2': 512, 'layer3': 1024, 'layer4': 2048}
        self.channel_slider.max = ch_map[change['new']] - 1
        self.trainer.reset_logs()

    def _on_loss_param_change(self, change):
        self.trainer.reset_logs()

    def _on_dd_toggle(self, change):
        self.channel_slider.disabled = change['new']
        self.trainer.reset_logs()

    def _on_projection_type_change(self, change):
        self.projector_type = change['new']
        self.trainer.reset_logs()
        self.init_system()

    def _on_zoom_change(self, change):
        new_w = change['new']
        self.img_widget.layout.width = f"{new_w}px"
        self.img_widget.layout.height = f"{new_w // 3}px"

    def _on_record_toggle(self, change):
        if change['new']:
            self.recorded_frames = []
            self.btn_record.description = '⏹️ Stop & Download'
            self.btn_record.button_style = 'danger'
        else:
            self.btn_record.description = '⏳ Processing...'
            self.btn_record.disabled = True
            if self.recorded_frames:
                try:
                    h, w, c = self.recorded_frames[0].shape
                    filename = 'nca_simulation.mp4'
                    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                    out_video = cv2.VideoWriter(filename, fourcc, 30.0, (w, h))
                    for frame in self.recorded_frames:
                        out_video.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
                    out_video.release()

                    from google.colab import files
                    files.download(filename)
                    self.stats_label.value = "💾 Recording exported successfully."
                except Exception as ev:
                    print("Video recording encoding error:", ev)
            self.btn_record.description = '🔴 Record Video'
            self.btn_record.button_style = 'info'
            self.btn_record.disabled = False
            self.recorded_frames = []

    def _on_play_stop_toggle(self, change):
        self.is_running = change['new']
        if self.is_running:
            self.btn_toggle.icon = 'stop'
            self.btn_toggle.button_style = 'danger'
            self._unpause_wall = time.time()
        else:
            self.btn_toggle.icon = 'play'
            self.btn_toggle.button_style = 'success'

    def _on_pause_toggle(self, change):
        self.is_paused = change['new']
        if self.is_paused:
            self._pause_offset = self.t
            self.btn_pause.icon = 'play'
        else:
            self._unpause_wall = time.time()
            self.btn_pause.icon = 'pause'

    def _on_hard_reset_click(self, change):
        self.init_system()
        self._update_graph_placeholder("Universe Destroyed. Rebuilding...")
        self.stats_label.value = "⚡ Physics Re-rolled | 🏆 Loss: inf"

    def reset_time(self):
        self.t = 0.0
        self._pause_offset = 0.0
        self._unpause_wall = time.time()
        self.display_x0 = self.task.generate_input(batch_size=1, channels=12, size=128, device=device)
        self.z_rk4 = self.model.lift(self.display_x0)

    def _update_graph_placeholder(self, text):
        fig, ax = plt.subplots(figsize=(6, 2.5))
        ax.text(0.5, 0.5, text, ha='center', va='center', color='gray')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        fig.tight_layout(pad=1.0)
        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight')
        plt.close(fig)
        self.graph_widget.value = buf.getvalue()

    def get_current_config(self):
        return {
            "target_layer": self.layer_dropdown.value,
            "target_channel": self.channel_slider.value,
            "is_deep_dream": self.dd_toggle.value
        }

    def render_panel(self):
        legend_label = widgets.HTML(
            value=f"<div style='display:flex; width:100%; text-align:center; font-family:sans-serif; font-size:14px; font-weight:bold; color:#ccc; background:#222; padding:6px 0; border-radius:4px 4px 0 0;'><div style='flex:1;'>⚡ Closed-Form O(1)</div><div style='flex:1;'>🦠 Euler Cellular Automaton</div><div style='flex:1;'>🔮 2D FFT Spectrum</div></div>"
        )

        viewport_ui = widgets.VBox([
            legend_label,
            self.img_widget,
            widgets.HBox([self.btn_toggle, self.btn_pause, self.btn_reset, self.zoom_slider, self.btn_record, self.fps_label],
                         layout=widgets.Layout(align_items='center', margin='5px 0'))
        ], layout=widgets.Layout(border='1px solid #333', padding='10px', border_radius='4px', background_color='#1a1a1a'))

        controls_styled = widgets.HBox([
            self.layer_dropdown, self.channel_slider, self.dd_toggle, self.proj_toggle, self.btn_hard_reset
        ], layout=widgets.Layout(border='1px solid #333', padding='10px', margin='5px 0', border_radius='4px', background_color='#1a1a1a', align_items='center'))

        layout_block = widgets.VBox([
            viewport_ui,
            controls_styled,
            widgets.HBox([self.graph_widget, self.stats_label],
                         layout=widgets.Layout(border='1px solid #333', padding='10px', border_radius='4px', background_color='#1a1a1a', align_items='center'))
        ], layout=widgets.Layout(padding='10px', background_color='#111', border_radius='8px'))

        display(layout_block)


# --- SECTION 6: RUNTIME PIPELINE INITIALIZATION ---

dashboard = Dashboard(
    model_class=ClosedFormNCA,
    task_class=CLIPFeatureTargetTask,
    trainer_class=NCATrainer
)
dashboard.render_panel()

last_render_time = 0.0
last_graph_update = 0.0
last_ui_update = 0.0
render_frames_history = deque(maxlen=30)


def colab_render_step():
    global last_render_time
    try:
        if not dashboard.is_running:
            return

        t_now = time.time()
        if not dashboard.is_paused:
            new_t = dashboard._pause_offset + (t_now - dashboard._unpause_wall) * 2.0
            dt = new_t - dashboard.t
            dashboard.t = new_t
        else:
            dt = 0.0

        with torch.no_grad():
            t_tensor = torch.tensor([dashboard.t], device=device).view(1, 1, 1, 1)
            _, rgb_cf = dashboard.model(t_tensor, dashboard.display_x0)

            sub_steps = max(1, int(dt / 0.1))
            for _ in range(sub_steps):
                dashboard.z_rk4 = dashboard.model.step_latent_euler(dashboard.z_rk4, dt=dt/sub_steps)

            rgb_cf_proj = rgb_cf[0].permute(1, 2, 0)
            rgb_rk4_proj = dashboard.model.project(dashboard.z_rk4)[0].permute(1, 2, 0)

            # Compute 2D FFT Magnitude Spectrum of the active Euler visualization
            gray = 0.2989 * rgb_rk4_proj[:, :, 0] + 0.5870 * rgb_rk4_proj[:, :, 1] + 0.1140 * rgb_rk4_proj[:, :, 2]
            Y = torch.fft.fft2(gray.float())
            Y_shifted = torch.fft.fftshift(Y)
            mag = torch.abs(Y_shifted)
            log_mag = torch.log1p(mag)
            log_mag_norm = (log_mag - log_mag.min()) / (log_mag.max() - log_mag.min() + 1e-5)

            fft_view = torch.zeros_like(rgb_rk4_proj)
            fft_view[:, :, 0] = log_mag_norm * 0.8  # Red
            fft_view[:, :, 1] = log_mag_norm * 0.2  # Green
            fft_view[:, :, 2] = log_mag_norm * 1.0  # Blue

            combined_spatial = torch.cat([rgb_cf_proj, rgb_rk4_proj, fft_view], dim=1)
            img_np = (combined_spatial.clamp(0, 1).mul_(255)).byte().cpu().numpy()

        dashboard.img_widget.value = cv2.imencode('.jpg', img_np[:, :, ::-1], [int(cv2.IMWRITE_JPEG_QUALITY), 75])[1].tobytes()

        if dashboard.btn_record.value:
            dashboard.recorded_frames.append(img_np.copy())
            if len(dashboard.recorded_frames) >= 900:
                dashboard.btn_record.value = False

        render_frames_history.append(t_now - last_render_time)
        dashboard.fps_label.value = f"FPS: {1.0/max(sum(render_frames_history)/len(render_frames_history), 1e-4):.0f} | t={dashboard.t:.2f}"
        last_render_time = t_now
    except Exception:
        print("\nRENDER ERROR:\n", traceback.format_exc())


def colab_train_step():
    global last_graph_update, last_ui_update
    try:
        if not dashboard.is_running or dashboard.is_paused:
            return

        t_now = time.time()
        config = dashboard.get_current_config()

        loss_breakdown = dashboard.trainer.step(config)

        if t_now - last_ui_update >= 0.3:
            train_times = dashboard.trainer.train_times
            it_s = 1.0 / max(sum(train_times) / len(train_times), 1e-4) if train_times else 0.0

            dashboard.stats_label.value = f"""
            <div style='font-family: monospace; font-size: 11px; color: #FFD700; background: #1a1a1a; padding: 10px; border-radius: 4px; border: 1px solid #333; line-height: 1.4;'>
              <div style='font-weight: bold; color: #FFF; margin-bottom: 5px; font-size: 12px;'>⚡ REAL-TIME METRICS (Train: {it_s:.0f} it/s)</div>
              <div>🏆 Best Feature Loss: <span style='color: #00FF00; font-weight: bold;'>{dashboard.trainer.best_loss:.4f}</span></div>
              <div style='color: #444; margin: 4px 0;'>---------------------------------------------</div>
              <div style='font-weight: bold; color: #aaa; margin-bottom: 3px;'>📊 Current Loss Components:</div>
              <div>  • Total Loss:        <span style='color: #1f77b4; font-weight: bold;'>{loss_breakdown['total']:.4f}</span></div>
              <div>  • Feature/Dream Loss: <span style='color: #ff7f0e; font-weight: bold;'>{loss_breakdown['feature']:.4f}</span></div>
              <div>  • L2 Reg Loss:       <span style='color: #2ca02c; font-weight: bold;'>{loss_breakdown['l2']:.4f}</span></div>
              <div>  • TV Reg Loss:       <span style='color: #d62728; font-weight: bold;'>{loss_breakdown['tv']:.4f}</span></div>
              <div>  • Activity/Sparsity: <span style='color: #9467bd; font-weight: bold;'>{loss_breakdown['sparsity']:.4f}</span></div>
            </div>
            """
            last_ui_update = t_now

        logs = dashboard.trainer.logs
        if len(logs["total"]) > 0 and (t_now - last_graph_update >= 3.0):
            fig, ax = plt.subplots(figsize=(6, 2.5))
            ax.plot(logs["total"][-500:], color='#1f77b4', linewidth=1.5, label='Total Loss')
            ax.plot(logs["feature"][-500:], color='#ff7f0e', linewidth=1.2, linestyle='--', label='Feature/Dream')
            ax.plot(logs["l2"][-500:], color='#2ca02c', linewidth=1.0, alpha=0.7, label='L2 Reg')
            ax.plot(logs["tv"][-500:], color='#d62728', linewidth=1.0, alpha=0.7, label='TV Reg')
            ax.plot(logs["sparsity"][-500:], color='#9467bd', linewidth=1.0, alpha=0.7, linestyle=':', label='Activity/Sparsity')
            ax.legend(loc='upper right', fontsize=8, framealpha=0.5)

            title_tag = f"Targeting {config['target_layer']}"
            if config['is_deep_dream']:
                title_tag += " (DeepDream)"
            else:
                title_tag += f" C{config['target_channel']}"

            ax.set_title(title_tag, fontsize=10)
            ax.grid(True, linestyle='--', alpha=0.5)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            fig.tight_layout(pad=1.0)

            buf = io.BytesIO()
            fig.savefig(buf, format='png', bbox_inches='tight')
            plt.close(fig)
            dashboard.graph_widget.value = buf.getvalue()
            last_graph_update = t_now
    except Exception:
         print("\nTRAIN ERROR:\n", traceback.format_exc())


def colab_perturb(x_norm, y_norm):
    try:
        h, w = 128, 128
        grid_y = int(y_norm * h)

        if x_norm < 0.33:
            grid_x = int((x_norm * 3.0) * w)
            Y, X = torch.meshgrid(torch.arange(h, device=device), torch.arange(w, device=device), indexing='ij')
            dist = (X - grid_x)**2 + (Y - grid_y)**2
            mask = (dist > 15**2).float().view(1, 1, h, w)
            dashboard.display_x0 = dashboard.display_x0 * mask
        elif x_norm < 0.66:
            grid_x = int(((x_norm - 0.33) * 3.0) * w)
            Y, X = torch.meshgrid(torch.arange(h, device=device), torch.arange(w, device=device), indexing='ij')
            dist = (X - grid_x)**2 + (Y - grid_y)**2
            mask = (dist > 15**2).float().view(1, 1, h, w)
            dashboard.z_rk4 = dashboard.z_rk4 * mask
    except Exception:
        print("\nPerturbation Error:\n", traceback.format_exc())


output.register_callback('notebook.colab_render_step', colab_render_step)
output.register_callback('notebook.colab_train_step', colab_train_step)
output.register_callback('notebook.colab_perturb', colab_perturb)

display(HTML("""
<style>
.colab-nca-viewport img {
    cursor: crosshair !important;
    border: 1px solid #444;
    border-radius: 2px;
    image-rendering: pixelated !important;
    image-rendering: crisp-edges !important;
}
</style>

<script>
(async function() {
    if (window.colab_render_interval) {
        clearInterval(window.colab_render_interval);
    }
    window.colab_train_active = false;
    await new Promise(resolve => setTimeout(resolve, 150));

    const viewportContainer = document.querySelector('.colab-nca-viewport');
    if (viewportContainer) {
        const img = viewportContainer.querySelector('img');
        if (img) {
            let isDrawing = false;

            const handlePaint = (e) => {
                const rect = img.getBoundingClientRect();
                const x = (e.clientX - rect.left) / rect.width;
                const y = (e.clientY - rect.top) / rect.height;
                if (x >= 0 && x <= 1 && y >= 0 && y <= 1) {
                    google.colab.kernel.invokeFunction('notebook.colab_perturb', [x, y], {});
                }
            };

            img.addEventListener('mousedown', (e) => {
                isDrawing = true;
                handlePaint(e);
            });

            img.addEventListener('mousemove', (e) => {
                if (isDrawing) {
                    handlePaint(e);
                }
            });

            window.addEventListener('mouseup', () => {
                isDrawing = false;
            });
            console.log("Mouse paint event listeners attached.");
        }
    }

    window.colab_train_active = true;
    console.log("Dual channels activated.");

    window.colab_render_interval = setInterval(() => {
        google.colab.kernel.invokeFunction('notebook.colab_render_step', [], {});
    }, 33);

    while (window.colab_train_active) {
        try {
            await google.colab.kernel.invokeFunction('notebook.colab_train_step', [], {});
        } catch (e) {
            console.error("Train pipeline error:", e);
            await new Promise(resolve => setTimeout(resolve, 1000));
        }
        await new Promise(resolve => setTimeout(resolve, 2));
    }
})();
</script>
"""))